In [ ]:
import os, random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:

DATA_DIR   = "/content"
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 10
LR         = 1e-3
NUM_WORKERS = 2

In [ ]:
# Download dataset (Colab + Kaggle API configured)
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset --unzip -p {DATA_DIR}

In [ ]:
def get_class_paths(path):
    classes, class_paths = [], []
    for label in sorted(os.listdir(path)):
        label_path = os.path.join(path, label)
        if os.path.isdir(label_path):
            for image in sorted(os.listdir(label_path)):
                classes.append(label)
                class_paths.append(os.path.join(label_path, image))
    return pd.DataFrame({"Class Path": class_paths, "Class": classes})

train_full_df = get_class_paths(os.path.join(DATA_DIR, "Training"))
ts_df         = get_class_paths(os.path.join(DATA_DIR, "Testing"))

class_names = sorted(train_full_df["Class"].unique())
class_to_idx = {c: i for i, c in enumerate(class_names)}
idx_to_class = {i: c for c, i in class_to_idx.items()}
num_classes = len(class_names)
print("Classes:", class_names)
print("Train(full):", len(train_full_df), "| Test:", len(ts_df))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.countplot(data=train_full_df, x="Class", ax=ax[0]); ax[0].set_title("Training class distribution")
sns.countplot(data=ts_df, x="Class", ax=ax[1]); ax[1].set_title("Test class distribution")
plt.tight_layout(); plt.show()

In [ ]:
# 80/20 stratified train/validation split (same ratio as the original validation_split=0.2)
tr_df, val_df = train_test_split(train_full_df, test_size=0.2,
                                 stratify=train_full_df["Class"], random_state=SEED)
tr_df, val_df = tr_df.reset_index(drop=True), val_df.reset_index(drop=True)
print(len(tr_df), len(val_df), len(ts_df))

## Data pipeline (OpenCV)

In [ ]:
def load_image(path, size=IMG_SIZE):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)

def augment(img):
    """Random rotation/shift/zoom/shear + horizontal flip, using cv2.warpAffine."""
    h, w = img.shape[:2]
    angle = random.uniform(-20, 20)
    scale = random.uniform(0.8, 1.2)
    tx, ty = random.uniform(-0.2, 0.2) * w, random.uniform(-0.2, 0.2) * h
    shear = random.uniform(-0.2, 0.2)

    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)   # 2x3
    M = np.vstack([M, [0, 0, 1]])
    S = np.array([[1, shear, -shear * h / 2], [0, 1, 0], [0, 0, 1]], dtype=np.float64)
    T = np.array([[1, 0, tx], [0, 1, ty], [0, 0, 1]], dtype=np.float64)
    A = (T @ S @ M)[:2]

    out = cv2.warpAffine(img, A, (w, h), flags=cv2.INTER_LINEAR,
                         borderMode=cv2.BORDER_REPLICATE)        # ~ fill_mode='nearest'
    if random.random() < 0.5:
        out = cv2.flip(out, 1)
    return out

class MRIDataset(Dataset):
    def __init__(self, df, train=False):
        self.paths  = df["Class Path"].tolist()
        self.labels = [class_to_idx[c] for c in df["Class"]]
        self.train  = train

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = load_image(self.paths[i])
        if self.train:
            img = augment(img)
        x = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0   # CHW in [0,1]
        return x, self.labels[i]

train_loader = DataLoader(MRIDataset(tr_df, train=True),  batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device == "cuda"))
val_loader   = DataLoader(MRIDataset(val_df),             batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device == "cuda"))
test_loader  = DataLoader(MRIDataset(ts_df),              batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device == "cuda"))

In [ ]:
# Preview augmented training images
ds = MRIDataset(tr_df, train=True)
plt.figure(figsize=(14, 14))
for i in range(16):
    x, y = ds[i]
    plt.subplot(4, 4, i + 1)
    plt.imshow(x.permute(1, 2, 0).numpy())
    plt.title(idx_to_class[y]); plt.axis("off")
plt.tight_layout(); plt.show()

## Model

In [ ]:
def conv_bn(cin, cout):
    return [nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU(inplace=True), nn.BatchNorm2d(cout)]

class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            *conv_bn(3, 32), nn.MaxPool2d(2),
            *conv_bn(32, 64), *conv_bn(64, 64), nn.MaxPool2d(2), nn.Dropout(0.25),
            *conv_bn(64, 128), *conv_bn(128, 128), nn.MaxPool2d(2), nn.Dropout(0.25),
            *conv_bn(128, 256), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.35),
            nn.Linear(128, num_classes),          # logits (softmax is inside the loss)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = BrainTumorCNN(num_classes).to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Training

In [ ]:
criterion = nn.CrossEntropyLoss()
# weight_decay ~ the original L2 regularisation on the dense layer
optimizer = torch.optim.Adamax(model.parameters(), lr=LR, weight_decay=1e-4)

def run_epoch(loader, train):
    model.train(train)
    total_loss, preds, trues = 0.0, [], []
    with torch.set_grad_enabled(train):
        for x, y in tqdm(loader, leave=False):
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            preds.append(out.argmax(1).cpu().numpy()); trues.append(y.cpu().numpy())
    preds, trues = np.concatenate(preds), np.concatenate(trues)
    p, r, _, _ = precision_recall_fscore_support(trues, preds, average="macro", zero_division=0)
    return {"loss": total_loss / len(loader.dataset),
            "accuracy": accuracy_score(trues, preds), "precision": p, "recall": r}

history = {f"{s}_{m}": [] for s in ("train", "val") for m in ("loss", "accuracy", "precision", "recall")}
best_val_loss, best_path = float("inf"), "best_brain_tumor_cnn.pth"

for epoch in range(1, EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    for m in tr: history[f"train_{m}"].append(tr[m]); history[f"val_{m}"].append(va[m])
    if va["loss"] < best_val_loss:
        best_val_loss = va["loss"]
        torch.save({"model_state": model.state_dict(), "epoch": epoch,
                    "class_names": class_names}, best_path)
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {tr['loss']:.4f} acc {tr['accuracy']:.4f} | "
          f"val loss {va['loss']:.4f} acc {va['accuracy']:.4f} "
          f"prec {va['precision']:.4f} rec {va['recall']:.4f}")

In [ ]:
# Training curves
metrics = ["accuracy", "loss", "precision", "recall"]
plt.figure(figsize=(20, 12))
for i, m in enumerate(metrics, 1):
    tr_m, va_m = history[f"train_{m}"], history[f"val_{m}"]
    best = int(np.argmin(va_m) if m == "loss" else np.argmax(va_m))
    plt.subplot(2, 2, i)
    ep = range(1, len(tr_m) + 1)
    plt.plot(ep, tr_m, "r", label=f"Training {m}")
    plt.plot(ep, va_m, "g", label=f"Validation {m}")
    plt.scatter(best + 1, va_m[best], s=150, c="blue", label=f"Best epoch = {best + 1}")
    plt.title(f"Training and Validation {m.title()}")
    plt.xlabel("Epochs"); plt.ylabel(m.title()); plt.legend(); plt.grid(True)
plt.suptitle("Model Training Metrics over epochs", fontsize=16)
plt.show()

## Evaluation

In [ ]:
# Load best checkpoint
ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print("Loaded best model from epoch", ckpt["epoch"])

@torch.no_grad()
def predict_loader(loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        ps.append(model(x.to(device)).argmax(1).cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(ys), np.concatenate(ps)

for name, loader in [("Train", train_loader), ("Valid", val_loader), ("Test", test_loader)]:
    r = run_epoch(loader, False)
    print(f"{name:5s} | accuracy {r['accuracy']*100:.2f}% | loss {r['loss']:.4f}")

In [ ]:
y_true, y_pred = predict_loader(test_loader)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Test-set Confusion Matrix"); plt.xlabel("Predicted Class"); plt.ylabel("True Class")
plt.tight_layout(); plt.show()

## Single-image prediction

In [ ]:
@torch.no_grad()
def predict(img_path):
    img = load_image(img_path)
    x = torch.from_numpy(img).permute(2, 0, 1).float().div(255).unsqueeze(0).to(device)
    model.eval()
    probs = torch.softmax(model(x), dim=1)[0].cpu().numpy()
    pred = int(probs.argmax())

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1); plt.imshow(img); plt.axis("off")
    plt.title(f"Input MRI Image\nPredicted: {class_names[pred]}")
    plt.subplot(1, 2, 2)
    bars = plt.barh(class_names, probs)
    plt.xlabel("Probability"); plt.title("Class Probabilities"); plt.xlim(0, 1)
    plt.bar_label(bars, fmt="%.2f")
    plt.tight_layout(); plt.show()
    print("Predicted tumor type:", class_names[pred])
    return class_names[pred], probs

predict(os.path.join(DATA_DIR, "Testing", "meningioma", "Te-meTr_0005.jpg"))

In [ ]:
torch.save({"model_state": model.state_dict(), "class_names": class_names,
            "class_to_idx": class_to_idx}, "brain_tumor_cnn_final.pth")
print("Saved brain_tumor_cnn_final.pth | classes:", idx_to_class)